# 06 — Model 2.2, typed flags with the class-aware freeze

**The model in math terms.** M2.2 is the canonical M2.1.1 verbatim — same states, same joint-alphabet emission, same fitted q-tables, same prediction layer — plus one rule on the transition. The chassis update (per home KC, belief $b_t$, the turn's symbol from the partitioned alphabet supplying the likelihood pair) is exactly notebook 04's; the addition is the gate, mounted as a drift-rate hook on the shared chassis.

**The gate.** Let $\mathcal{F}_{k,t}$ be the set of bias-class flags homed to KC $k$ that have fired for this student at or before turn $t$. The learn rate becomes

$$\tau_k(t) = \begin{cases} 0 & \text{if } \mathcal{F}_{k,t} \neq \varnothing \\ \tau_k & \text{otherwise} \end{cases} \qquad b^{\text{pre}}_t = b_{t-1} + (1 - b_{t-1})\,\tau_k(t)$$

First fire, ratcheted for the session, never unfrozen. Evidence still updates belief through Bayes — quiets and corrects still lift it; what dies post-fire is the free upward drift, improvement the model was never shown.

**The class table.** Bias-class (conjunction, inverse, time-axis, base-rate neglect): fires freeze the home KC — kc1, kc2, kc5 are freezable, and on kc2 either face trips it. Skill-class (denominator neglect): fires never gate — kc4's drift always runs.

**Grounding.** The CPR instruction-resistance table (two weeks of teaching moved conjunction 21→24% correct, inverse 35→35, time-axis 37→25, while the denominator-hosting skill jumped 18→69) and the impasse account of self-repair (skill gaps are felt and repaired through practice; biases are walked away from confidently). In a session with no feedback, letting a bias silently improve is inventing evidence.

**Zero new parameters.** The class assignment is legislated from the literature, the freeze is a hard zero, and every fitted quantity (chains, u-tables, anchors, w_mix) is estimated exactly as in the chassis — the q-tables under their Jeffreys shrink ($\kappa_Q = 1$ toward $0.5$), exactly as in notebook 04. M2.2 minus M2.1.1 is therefore a pure test of the class-dependent persistence claim, the design's novel contribution, and the registered tested secondary.

**File layout.**
* The model: `scripts/model_2_2.py` (subclasses `Model_2_1_1`; overrides only the transition)
* Comparisons: **loaded from stored predictions** — `cache/model_2_1_1/Model_2_1_1/predictions.csv` and `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`; neither is re-run here
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Harness `scripts/evaluator.py`, data `data/data_annotated.csv`, loader `scripts/data.py`
* Saved outputs: `cache/model_2_2/Model_2_2/` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [1]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_2 import Model_2_2
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
M12_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'
M211_PREDS = 'cache/model_2_1_1/Model_2_1_1_Joint/predictions.csv'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
m12 = pd.read_csv(M12_PREDS)
m211 = pd.read_csv(M211_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(m12), 'and', len(m211), 'stored comparison predictions')

312 rows | 26 cached folds | 312 and 312 stored comparison predictions


## 1. Run
M2.2 through the shared harness with the cached inner chains. Both comparisons come from stored predictions, never re-fitted.

In [2]:
ev = Evaluator(Model_2_2, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored runs
Same references as the earlier notebooks (qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.653).

In [3]:
pd.DataFrame([dict(model='M2.2 (class-aware gate)', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M2.1.1 (chassis, stored)', **{k: round(float(v),4) for k,v in _metrics(m211.y_true, m211.p_pred).items()}),
              dict(model='M1.2 (flag-blind, stored)', **{k: round(float(v),4) for k,v in _metrics(m12.y_true, m12.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
M2.2 (class-aware gate),0.6980,0.5787,0.6266,0.5961,0.7051,0.7974,312.0
"M2.1.1 (chassis, stored)",0.6870,0.5827,0.6182,0.6029,0.7019,0.7974,312.0
"M1.2 (flag-blind, stored)",0.6741,0.5747,0.6018,0.6070,0.6859,0.7860,312.0


### 2.2 Where the gain lives
The gate's designed disagreement set is every same-KC turn after a student's first bias-class fire. The split shows the contrast there against everywhere else, then the per-participant deltas and the row-level helps and hurts. First bias fires: P23 Q3; P02, P03, P11, P20, P24 Q4; P06 Q6. P01 (denominator only, skill-class) never trips the gate.

In [4]:
j = preds.merge(m211, on=['participant_id','question_number'], suffixes=('_g','_b'))
first_bias = {'P02': 4, 'P03': 4, 'P06': 6, 'P11': 4, 'P20': 4, 'P23': 3, 'P24': 4}
j['postfire'] = j.apply(lambda r: r.participant_id in first_bias
                        and r.question_number > first_bias[r.participant_id], axis=1)
rows = []
for lbl, sub in (('post-first-bias-fire', j[j.postfire]), ('all other rows', j[~j.postfire])):
    rows.append(dict(rows=lbl, n=len(sub),
                     gate=round(_metrics(sub.y_true_g, sub.p_pred_g)['auc'], 3),
                     chassis=round(_metrics(sub.y_true_b, sub.p_pred_b)['auc'], 3)))
pd.DataFrame(rows).set_index('rows')

,n,gate,chassis
rows,,,
post-first-bias-fire,55,0.611,0.531
all other rows,257,0.662,0.660


In [5]:
j['good'] = np.where(j.y_true_g == 1, j.p_pred_g - j.p_pred_b, j.p_pred_b - j.p_pred_g)
print(f'rows helped (good > 0.01): {int((j.good > 0.01).sum())} | rows hurt: {int((j.good < -0.01).sum())}')
print('biggest helps:')
display(j.nlargest(6, 'good')[['participant_id','question_number','p_pred_b','p_pred_g','y_true_g']].round(3))
print('biggest hurts:')
display(j.nsmallest(6, 'good')[['participant_id','question_number','p_pred_b','p_pred_g','y_true_g']].round(3))

rows helped (good > 0.01): 106 | rows hurt: 70
biggest helps:


,participant_id,question_number,p_pred_b,p_pred_g,y_true_g
268,P23,5,0.722,0.460,0
286,P24,11,0.485,0.261,0
125,P11,6,0.596,0.404,0
17,P02,6,0.571,0.387,0
280,P24,5,0.581,0.407,0
28,P03,5,0.600,0.432,0


biggest hurts:


,participant_id,question_number,p_pred_b,p_pred_g,y_true_g
233,P20,6,0.583,0.390,1
124,P11,5,0.561,0.414,1
16,P02,5,0.553,0.410,1
129,P11,10,0.576,0.444,1
285,P24,10,0.308,0.227,1
287,P24,12,0.237,0.166,1


In [6]:
rows = []
for pid, g in j.groupby('participant_id'):
    a1 = _metrics(g.y_true_b, g.p_pred_b)['auc']; a2 = _metrics(g.y_true_g, g.p_pred_g)['auc']
    if a1 == a1 and a2 == a2:
        rows.append(dict(participant=pid, chassis=round(a1,3), gate=round(a2,3), delta=round(a2-a1,3)))
pd.DataFrame(rows).sort_values('delta').set_index('participant')

,chassis,gate,delta
participant,,,
P20,0.861,0.778,-0.083
P01,0.667,0.667,0.000
P22,0.818,0.818,0.000
P21,0.909,0.909,0.000
P19,0.364,0.364,0.000
P18,0.909,0.909,0.000
P16,0.450,0.450,0.000
P15,0.750,0.750,0.000
P13,0.625,0.625,0.000


### 2.3 Fitted tables and bridge anchors
The u-tables are fitted exactly as in the chassis (kappa 5); the anchors are refit on gated walks, and g0 is where the gate's cost shows.

In [7]:
q0 = pd.DataFrame([m.q0 for m in ev.fold_models.values()]).mean().round(3)
print('fold-mean q0 (fired | wrong, presented, unmastered):')
print(q0.to_string())
print()
print('anchors: s0', round(float(np.mean([m.s0 for m in ev.fold_models.values()])),3),
      '| g0', round(float(np.mean([m.g0 for m in ev.fold_models.values()])),3),
      '| censuses 0.077 / 0.061')

fold-mean q0 (fired | wrong, presented, unmastered):
conjunction            0.126
inverse                0.498
time_axis              0.738
denominator_neglect    0.665
base_rate_neglect      0.430

anchors: s0 0.081 | g0 0.141 | censuses 0.077 / 0.061


### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, the gate beside the stored chassis.

In [8]:
def cmat(p):
    yhat = (p.p_pred >= 0.5).astype(int)
    cm = pd.crosstab(p.y_true.map({1:'actual correct', 0:'actual wrong'}),
                     yhat.map({1:'predicted correct', 0:'predicted wrong'}))
    return cm.reindex(index=['actual correct','actual wrong'],
                      columns=['predicted correct','predicted wrong'], fill_value=0)

print('M2.2:'); display(cmat(preds))
print('M2.1.1 (stored):'); display(cmat(m211))

M2.2:


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,181,19
actual wrong,73,39


M2.1.1 (stored):


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,183,17
actual wrong,76,36


## 3. Conclusion

* **The tested secondary lands positive: the class-aware gate beats its chassis on every headline metric.** On the joint canonical mounting M2.2 reaches AUC ~0.698 against the chassis's ~0.687 (log-loss ~0.596 vs ~0.603), on the same 312 targets with identical chains and q-tables — a one-line, zero-parameter intervention. Exact values are this run's; read them from section 2.1.
* **The attribution is the cleanest in the family.** On the designed disagreement set — the rows after a student's first bias-class fire — the gate's edge concentrates (~0.61 vs ~0.53 on this mounting) while the remaining rows sit flat. The entire edge lives exactly where the mechanism operates and nowhere else; section 2.2 prints this run's split.
* **The class exclusion is verified at full scale.** P01, the only skill-class-only firer, sits at exactly zero AUC delta: her denominator fires never trip the gate and kc4's drift keeps running, the legislated asymmetry doing precisely and only what it claims.
* **The per-participant ledger splits, and not along the crash axis.** Winners and losers both appear among the ever-fired (section 2.2's table carries this run's values); recoverers can gain under the freeze, so freeze-versus-crash and persist-versus-recover are different axes: the gate withholds optimism without deepening the crash.
* **The bridge carries any cost.** Read s0 and g0 against their censuses (0.077 / 0.061) from section 2.3's printout; on the factorized mounting the cost surfaced as g0 inflation, frozen lows pushing realized corrects through the guess anchor, and I check the same channel here rather than assert it.
* **The result is mounting-robust.** The gate's designed-set edge survived the emission-form switch essentially intact (about +0.077 on the joint chassis against +0.071 on the registered factorized record), so the dynamics claim does not lean on the emission form.
* **Caveats.** Participant-clustered bootstrap intervals are deferred until the family closes; the designed set is small; the cell-layer check runs separately; and the grounding predicts the direction, not the size — the effect rides on kc2's large fitted learn rate (~0.42), which is where most of the frozen drift lived.

## 4. Save
Persist the run: per-fold bridge, shape, and q-tables, the pooled predictions, the metrics, and the index carrying the class table.

In [9]:
import os
import json
from scripts.model_2_2 import save_model_2_2_from_evaluator

out_dir = 'cache/model_2_2'
save_model_2_2_from_evaluator(ev, out_dir)

'cache/model_2_2/Model_2_2'